# GSB 5544 — Topic 4.1: Text as Data  
*Fill each `____` blank as you work; the ✅ checks ask for a sentence or two.*

## The next 20 minutes

| | Question | Where it lands in PA 4.1 |
|---|---|---|
| **a. Represent** | How do we turn free text into rows and columns? | parts 1–3 |
| **b. Weight** | Why downweight words that appear everywhere? (TF-IDF) | parts 4–5 |
| **c. Compare** | How do we measure "these two documents are similar"? | parts 6–11 |

Week 3 asked *which rows of a table are most alike?* and answered it with a **distance**.
This week the observations are **emails** — no columns at all, just text. The whole trick of
text analysis is: **build the columns yourself**, then everything from Week 3 works again.

In [ ]:
import pandas as pd
import numpy as np

---
## 1. The vocabulary of text analysis

Three words you will see in every reading on this topic:

| Term | Meaning | In PA 4.1 |
|---|---|---|
| **document** | one unit of text — one email, one review, one tweet | one email body |
| **corpus** | the collection of documents | the `body` column |
| **bag of words** | represent a document by *which words it uses and how often*, ignoring word order | what `CountVectorizer` builds |

"Bag" is the honest part of the name: *"the dog bit the man"* and *"the man bit the dog"*
become the **same** bag. We throw away order and keep counts — a crude summary that turns
out to be surprisingly powerful for questions like "is this spam?"

---
## 2. The term-frequency (TF) matrix — by hand first

A tiny corpus of four "emails". Before running anything: what should the columns of our
table be, and what number goes in each cell?

In [ ]:
corpus = pd.Series([
    "Win a FREE prize now!!",
    "Meeting moved to noon.",
    "Free free FREE — claim your prize",
    "Can we move the meeting?",
])
corpus

**The recipe** (this is the answer to PA 4.1 part 2):

1. **Normalize** — lowercase everything, so `Free`, `free`, and `FREE` are one word.
2. **Tokenize** — split each document into words (*tokens*), dropping punctuation.
3. **Build the vocabulary** — the set of distinct words across the whole corpus. These are the **columns**.
4. **Count** — cell (i, j) = how many times word j appears in document i. These counts are the **term frequencies (TF)**.

The result is called the **term-frequency matrix** (or *document-term matrix*):
one row per document, one column per vocabulary word. Text is now a table, and
pandas is back in business.

`scikit-learn` wraps all four steps in one object, `CountVectorizer`:

In [ ]:
from sklearn.feature_extraction.text import ____

vec = ____()
tf = vec.____(corpus)               # learn the vocabulary, then count
tf_df = pd.DataFrame(tf.____(),           # sparse matrix -> regular array
                     columns=vec.____())
tf_df

Read one row aloud to check it: document 2 (*"Free free FREE — claim your prize"*)
has `free = 3`, `prize = 1`, `claim = 1`, and 0 everywhere else. Word order is gone;
the counts remain.

✅ **Check:** why does document 0's row show `win = 1` and not `Win = 1`? And why is
`fit_transform` one step, not two separate calls here?

**Your answer:** *(write it here — replace this line)*

---
## 3. The problem with raw counts — and the TF-IDF fix

Look down the columns of a real TF matrix and two kinds of words appear:

- **Words that show up in almost every document** — `the`, `to`, `and`. Big counts, zero information: knowing an email contains `the` tells you nothing about it.
- **Words that show up in a few documents** — `prize`, `viagra`, `invoice`. Small counts, big signal.

Raw TF rewards exactly the wrong words. The fix weights each column by how *rare* the word is
across the corpus:

| Quantity | Definition | Intuition |
|---|---|---|
| **TF** — term frequency | count of word *j* in document *i* | "how much does *this* document use the word?" |
| **DF** — document frequency | number (or fraction) of documents containing word *j* | "how common is the word across the corpus?" |
| **IDF** — inverse document frequency | roughly `log(n_documents / DF)` | big for rare words, near 0 for words in every document |
| **TF-IDF** | TF × IDF | large only when a word is *frequent in this document* **and** *rare overall* |

A word that appears in **every** document has IDF ≈ log(1) = 0 — its column is wiped out,
no matter how large its counts. `TfidfVectorizer` is a drop-in replacement for `CountVectorizer`:

In [ ]:
from sklearn.feature_extraction.text import ____

tfidf_vec = ____()
tfidf_df = pd.DataFrame(tfidf_vec.____(corpus).toarray(),
                        columns=tfidf_vec.get_feature_names_out())
tfidf_df.round(2)

✅ **Check:** look at row 0. In the TF matrix, `win`, `free`, and `prize` all had count 1 —
yet in the TF-IDF matrix `win` scores **higher** than the other two. Why?

**Your answer:** *(write it here — replace this line)*

---
## 4. Comparing documents: cosine similarity, not Euclidean distance

Each email is now a row of numbers — a **vector**. Week 3's instinct says: Euclidean distance.
For text there is a catch: **document length**. A 2,000-word email uses every one of its words
more often than a 50-word email about the *same topic*, so Euclidean distance calls them far apart
just because one is longer.

The fix: compare the **direction** of the two vectors, not their length.

$$\text{cosine similarity}(x, y) = \frac{x \cdot y}{\lVert x \rVert \, \lVert y \rVert}
\qquad\qquad \text{cosine distance} = 1 - \text{cosine similarity}$$

| Value | Meaning |
|---|---|
| similarity ≈ 1 (distance ≈ 0) | same mix of words in the same proportions |
| similarity ≈ 0 (distance ≈ 1) | no overlapping words at all (counts can't be negative, so ≈ 0 is the floor here) |

Scaling a document up — writing the same email twice as long — doesn't change its direction,
so cosine similarity ignores length and keeps topic. That is exactly what we want.

In [ ]:
from sklearn.metrics.pairwise import ____

sim = ____(tf_df)                       # 4×4: every document vs every document
pd.DataFrame(sim, columns=corpus.str[:20], index=corpus.str[:20]).round(2)

✅ **Check:** which pair of documents is most similar, and does that match your reading of the
four sentences? Which document is *"Can we move the meeting?"* closest to?

**Your answer:** *(write it here — replace this line)*

---
## 5. The payoff: classify by "who are your neighbours?"

Now stack the pieces exactly the way PA 4.1 will:

1. Corpus → **TF (or TF-IDF) matrix** — text becomes a table.
2. A new email arrives with **unknown** label.
3. Compute the **cosine distance** from the new email to every email whose label we know.
4. **Sort.** Look at the closest emails — if the nearest neighbours are spam, bet spam.

That is nearest-neighbour classification, built entirely from this notebook's parts.
A dry run on the real PA data (a sample of the Enron corpus — read the PA intro for the story):

In [ ]:
emails = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/refs/heads/main/data/enron_email.csv")
emails["spam"].value_counts(dropna=False)

1,980 emails labelled spam (1) or not (0) — and **10 emails with no label**. PA 4.1's job is to
guess those 10 labels using nearest neighbours. One caution before you start: a few `body`
entries are missing, and `CountVectorizer` refuses `NaN` documents — `.fillna("")` first.

In [ ]:
corpus = emails["body"].____               # CountVectorizer can't digest NaN
tf_real = CountVectorizer().fit_transform(corpus)
tf_real.shape

Nearly 35,000 columns — one per distinct word in 1,990 emails. You will never look at this
matrix directly; you compute with it.

## The three lines to keep

| | |
|---|---|
| **Represent** | `CountVectorizer()` / `TfidfVectorizer()` + `fit_transform(corpus)` → documents become rows, words become columns |
| **Weight** | TF-IDF = TF × IDF — downweights words that appear in most documents, keeps distinctive ones |
| **Compare** | `cosine_similarity(...)`; distance = 1 − similarity; sort and read the nearest neighbours |

PA 4.1 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).